In [23]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

with open("../top_100_universe.txt", "r") as f:
    ACTIVE_SET = [line.strip() for line in f]
START = "2023-06-01"
END = "2025-06-01"
DATASET = "XNAS.ITCH"

df_bars = (
    adapter.scan_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
    .select(["ts_event", "symbol", "close"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
    .filter(
        pl.col("symbol").is_in(ACTIVE_SET)
    )
    .sort(["ts_event", "symbol"])
)

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] scan_parquet: 548cfb7cc98703d7d8a8a14fe79dfe1a.parquet


In [24]:
CAX = (
     pl.read_csv("../CAX/corporate_actions.csv")
     .sort(["date", "symbol"])
     .with_columns(
         pl.col("date").str.to_datetime(time_unit="ns", time_zone="UTC"),
         pl.col("symbol").cast(pl.Categorical)
     )
)
CAX

symbol,date,dividend,split
cat,"datetime[ns, UTC]",f64,f64
"""AGG""",2023-06-01 00:00:00 UTC,0.258,0.0
"""AJG""",2023-06-01 00:00:00 UTC,0.55,0.0
"""ALL""",2023-06-01 00:00:00 UTC,0.89,0.0
"""ANGL""",2023-06-01 00:00:00 UTC,0.131,0.0
"""BAC""",2023-06-01 00:00:00 UTC,0.22,0.0
…,…,…,…
"""FIS""",2026-06-11 00:00:00 UTC,0.44,0.0
"""FNV""",2026-06-11 00:00:00 UTC,0.44,0.0
"""JXN""",2026-06-11 00:00:00 UTC,0.9,0.0


In [25]:
daily_px = (
      df_bars
      .collect(engine="streaming")
      .filter(pl.col("symbol").is_in(ACTIVE_SET))
      .join(CAX, left_on=["ts_event", "symbol"], right_on=["date", "symbol"], how="left")
      .sort(["symbol", "ts_event"])
      .with_columns(
          pl.col("dividend").fill_null(0.0),
          # 0.0 / null == "no split" -> ratio of 1.0, not 0.0
          pl.when(pl.col("split").fill_null(0.0) == 0.0)
            .then(1.0).otherwise(pl.col("split")).alias("split"),
      )
      .with_columns(
          (pl.col("split") / (1 - pl.col("dividend") / pl.col("close").shift().over("symbol"))).alias("CAX")
      )
      .with_columns(pl.col("CAX").fill_null(1.0))            # first row per symbol: no prev_close
      .with_columns(pl.col("CAX").cum_prod().over("symbol"))
  )
daily_px

ts_event,symbol,close,dividend,split,CAX
"datetime[ns, UTC]",cat,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,"""AAPL""",180.58,0.0,1.0,1.0
2023-06-02 00:00:00 UTC,"""AAPL""",181.1,0.0,1.0,1.0
2023-06-05 00:00:00 UTC,"""AAPL""",179.12,0.0,1.0,1.0
2023-06-06 00:00:00 UTC,"""AAPL""",179.04,0.0,1.0,1.0
2023-06-07 00:00:00 UTC,"""AAPL""",177.9,0.0,1.0,1.0
…,…,…,…,…,…
2025-05-23 00:00:00 UTC,"""ZS""",253.99,0.0,1.0,1.0
2025-05-27 00:00:00 UTC,"""ZS""",255.9,0.0,1.0,1.0
2025-05-28 00:00:00 UTC,"""ZS""",255.52,0.0,1.0,1.0


In [26]:
adj_px = (
    daily_px
    .with_columns((pl.col("close") * pl.col("CAX")).alias("adj_close"))
    .select(["ts_event", "symbol", "adj_close"])
)
adj_px

ts_event,symbol,adj_close
"datetime[ns, UTC]",cat,f64
2023-06-01 00:00:00 UTC,"""AAPL""",180.58
2023-06-02 00:00:00 UTC,"""AAPL""",181.1
2023-06-05 00:00:00 UTC,"""AAPL""",179.12
2023-06-06 00:00:00 UTC,"""AAPL""",179.04
2023-06-07 00:00:00 UTC,"""AAPL""",177.9
…,…,…
2025-05-23 00:00:00 UTC,"""ZS""",253.99
2025-05-27 00:00:00 UTC,"""ZS""",255.9
2025-05-28 00:00:00 UTC,"""ZS""",255.52


In [27]:
daily_ret = (
    adj_px
    .with_columns(
        pl.col("adj_close").pct_change().over("symbol").alias("ret")
    )
    .pivot(
        on="symbol",
        on_columns=ACTIVE_SET,
        index="ts_event",
        values="ret"
    )
    .with_columns(
        pl.all().exclude("ts_event").fill_null(pl.all().exclude("ts_event").mean())
    )
    .with_columns(
        (pl.all().exclude("ts_event") - pl.all().exclude("ts_event").mean()).round(6)
    )
)
daily_ret

ts_event,NVDA,TSLA,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,AMD,COST,COIN,APP,MU,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,BKNG,MRVL,AMAT,TMUS,PEP,INTU,ISRG,LRCX,CRWV,DASH,GILD,…,WDAY,EQIX,TEM,ON,TEAM,ADSK,MAR,DDOG,ROST,CHTR,FSLR,KDP,CSX,TTWO,RGTI,ZS,AEP,AXON,CTAS,EXC,EXE,MDB,ROP,EBAY,MNST,BKR,ITCI,EXPE,FANG,ULTA,WDC,IBKR,CTSH,OKTA,TER,IDXX,SOFI
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,-0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-06-02 00:00:00 UTC,-0.021408,0.026902,0.00249,0.004946,0.006565,-0.004621,0.007726,0.043334,-0.008817,-0.000188,0.007765,-0.011219,-0.020249,0.00627,0.014123,0.004703,0.001497,-0.042313,-0.003653,-0.003304,-0.044435,0.003118,0.008344,0.023769,0.023035,0.024091,-0.0215,0.000356,-0.063681,0.008266,0.01907,0.006028,-0.013176,0.0,-0.001252,0.022483,…,-0.016704,0.007731,-0.0,-0.010682,0.024167,0.008913,0.033753,-0.000788,0.021528,0.015395,-0.01151,0.010535,0.016067,-0.002311,-0.114918,0.068064,0.007553,0.000531,0.013819,0.008213,0.0,0.037776,0.004021,0.022748,0.003278,0.033109,0.039322,0.049253,0.02846,0.042894,0.035206,0.02132,0.025515,-0.011568,-0.00903,0.000287,0.034804
2023-06-05 00:00:00 UTC,-0.009008,0.008421,-0.011323,0.000288,0.006263,-0.008301,0.00974,-0.014004,0.054365,-0.086325,0.010169,0.003448,0.001576,0.008982,-0.098401,0.034828,-0.023388,-0.004415,-0.027608,-0.032469,0.012681,0.002703,-0.053782,-0.033898,-0.006,0.019152,-0.040247,-0.014688,0.008408,0.00494,0.024533,0.014036,-0.014585,0.0,0.018176,-0.003326,…,0.004582,-0.005419,-0.0,-0.015778,0.007544,0.010768,-0.003008,0.013354,0.003483,0.000621,-0.01329,-0.024359,0.008602,-0.005224,0.039175,0.036438,0.006647,-0.016581,-0.000347,0.007379,0.0,0.028606,0.007169,0.022409,-0.003101,0.002585,0.021701,0.027525,-0.009538,-0.007023,-0.035647,-0.002575,-0.037734,-0.026349,-0.009899,-0.019397,0.030632
2023-06-06 00:00:00 UTC,-0.008403,0.03989,-0.000836,-0.007403,0.010126,0.00153,0.009549,-0.021866,0.00906,0.072786,0.009495,-0.009208,0.059541,-0.003343,-0.09896,-0.10432,0.000812,0.058019,0.031883,0.001779,-0.00234,-0.00807,0.041037,0.002806,-0.002603,0.011442,0.029326,0.00315,-0.023548,-0.015178,0.010354,-0.006676,0.008665,0.0,0.028424,-0.020688,…,0.005634,-0.014937,-0.0,0.038835,-0.015938,-0.001084,0.014593,0.013655,0.00651,-0.012654,-0.033516,0.005824,-0.008245,-0.007872,0.223838,0.036988,-0.00543,0.025201,-0.004674,-0.002733,0.0,-0.001454,-0.002411,-0.003719,-0.01907,0.025485,0.052493,0.050141,-0.002331,-0.004422,-0.001489,0.006745,0.015241,0.013409,0.00614,-0.005545,0.061357
2023-06-07 00:00:00 UTC,-0.029175,-0.010576,-0.006757,-0.032833,-0.041243,-0.034214,-0.037731,0.003366,-0.065435,-0.040639,-0.039038,-0.002475,-0.054513,-0.012039,0.001993,-0.083974,-0.006868,0.011281,-0.010701,0.012303,-0.031776,-0.001098,0.009376,0.015905,-0.034265,-0.028892,-0.002328,0.009274,-0.014644,-0.008878,-0.056721,-0.027957,-0.000493,0.0,-0.023557,-0.004295,…,-0.053783,-0.004949,-0.0,0.008519,-0.06371,-0.026835,-0.015427,-0.078549,-0.028486,0.020073,-0.026486,0.002566,0.019241,-0.01984,-0.027942,-0.059129,0.02158,0.003226,-0.001523,0.011616,0.0,-0.046238,-0.010741,-0.009693,-0.025938,0.006756,-0.029354,-0.045434,0.026802,-0.004365,0.024139,-0.005282,0.000883,-0.014546,0.017128,-0.029963,0.000419
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-05-23 00:00:00 UTC,-0.015441,-0.008513,-0.028189,-0.008

In [28]:
# causal_vol = daily_ret.with_columns(pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1))
# st_ret = daily_ret.select(pl.all().exclude("ts_event")) / causal_vol.select(pl.all().exclude("ts_event"))
# st_ret


# floor the vol so a collapsed forecast can't produce a divide-by-~0
# (cross-sectional: e.g. 1st percentile of that day's vols, or a small abs floor)
causal_vol = daily_ret.with_columns(
    pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1)
)
CAUSAL_VOL_FLOOR = 1e-3   # ~0.1%/day
st_ret = (
    daily_ret.select(pl.all().exclude("ts_event"))
    / causal_vol.select(pl.all().exclude("ts_event")).select(pl.all().clip(lower_bound=CAUSAL_VOL_FLOOR))
).select(pl.all().clip(-10, 10))   # a >10σ standardized daily return is a data artifact

st_ret


NVDA,TSLA,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,AMD,COST,COIN,APP,MU,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,BKNG,MRVL,AMAT,TMUS,PEP,INTU,ISRG,LRCX,CRWV,DASH,GILD,AMGN,…,WDAY,EQIX,TEM,ON,TEAM,ADSK,MAR,DDOG,ROST,CHTR,FSLR,KDP,CSX,TTWO,RGTI,ZS,AEP,AXON,CTAS,EXC,EXE,MDB,ROP,EBAY,MNST,BKR,ITCI,EXPE,FANG,ULTA,WDC,IBKR,CTSH,OKTA,TER,IDXX,SOFI
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
-10.0,10.0,2.49,4.946,6.565,-4.621,7.726,10.0,-8.817,-0.188,7.765,-10.0,-10.0,6.27,10.0,4.703,1.497,-10.0,-3.653,-3.304,-10.0,3.118,8.344,10.0,10.0,10.0,-10.0,0.356,-10.0,8.266,10.0,6.028,-10.0,0.0,-1.252,10.0,10.0,…,-10.0,7.731,-0.0,-10.0,10.0,8.913,10.0,-0.788,10.0,10.0,-10.0,10.0,10.0,-2.311,-10.0,10.0,7.553,0.531,10.0,8.213,0.0,10.0,4.021,10.0,3.278,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,-10.0,-9.03,0.287,10.0
-0.586075,0.435993,-6.333781,0.081103,1.328766,-2.502046,1.755922,-0.450116,8.588146,-10.0,1.824054,0.428069,0.108406,1.995292,-9.704506,10.0,-10.0,-0.145331,-10.0,-10.0,0.397493,1.207455,-8.977668,-1.986388,-0.362797,1.107287,-2.60733,-10.0,0.183901,0.832401,1.791847,3.243179,-1.541785,0.0,10.0,-0.206048,1.448589,…,0.382063,-0.976302,-0.0,-2.057312,0.43479,1.682721,-0.124127,10.0,0.225346,0.056184,-1.608239,-3.220518,0.745702,-3.148503,0.474812,0.745655,1.225765,-10.0,-0.034975,1.251401,0.0,1.054732,2.483278,1.372082,-1.317631,0.108747,0.768679,0.778387,-0.466792,-0.228048,-1.410286,-0.168225,-2.059862,-3.172537,-1.526878,-10.0,1.225877
-0.624055,2.445601,-0.121337,-2.589461,1.905485,0.272552,1.305882,-0.829662,0.276635,1.416811,1.261014,-1.355991,5.079902,-0.517938,-1.673776,-4.99057,0.058228,2.362836,1.924781,0.091623,-0.087556,-3.343956,1.267891,0.115031,-0.189143,0.636535,1.087223,0.360271,-0.635026,-2.709093,0.566356,-0.73783,0.750977,0.0,2.621936,-1.576538,-0.295684,…,0.562487,-2.713692,-0.0,3.457605,-1.088003,-0.131929,0.746179,1.715427,0.516953,-1.423345,-3.245327,0.370504,-0.778574,-2.327101,3.184728,0.824506,-0.922604,2.552698,-0.586009,-0.423025,0.0,-0.052574,-0.496461,-0.198756,-7.21663,1.329758,2.010748,1.528641,-0.134164,-0.176179,-0.050696,0.544112,0.567605,0.78682,0.780728,-0.480313,2.262308
-2.374099,-0.421797,-1.147544,-7.175981,-5.917776,-7.053381,-4.724116,0.133538,-2.311245,-0.700386,-4.810636,-0.328699,-1.666557,-2.085711,0.027588,-1.462794,-0.577506,0.305997,-0.489721,0.742308,-1.393067,-0.233817,0.268242,0.763154,-2.901946,-1.756619,-0.084235,1.215202,-0.431731,-0.958217,-3.438597,-3.302533,-0.04553,0.0,-1.345657,-0.27598,0.244862,…,-5.955468,-0.543016,-0.0,0.379335,-4.243756,-3.819651,-0.841506,-7.973403,-2.529876,1.99509,-1.350069,0.18677,1.923531,-3.946767,-0.212443,-1.379896,3.743643,0.206275,-0.210826,2.041946,0.0,-1.96068,-2.482443,-0.603357,-2.536221,0.320436,-0.830556,-1.185218,1.803942,-0.202873,0.963872,-0.474211,0.036437,-0.901835,2.3045,-2.921048,0.010593
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.400951,-0.185018,-0.997228,-0.414814,-0.435369,-0.544211,-0.777881,-0.401753,-0.047433,-2.020114,-0.787855,-0.267392,-0.173723,-0.861658,-0.828469,-0.014069,-0.481423,-0.499992,-0.579086,-0.799101,0.667921,-0.309043,-0.69326,-0.56203,-0.770736,-0.15557,-0.410263,-0.525021,0.27871,-0.296493,-0.026589,-0.098666,-0.577629,-0.14067,0.609632,-0.167485,-0.160678,…,-2.342287,-0.814245

In [29]:
null_columns = [
    col for col in st_ret.columns 
    if st_ret.select(pl.col(col).is_null().all()).item()
]
null_columns

[]

In [8]:
sym_cols = [c for c in daily_ret.columns if c != "ts_event"]

X = (
    st_ret
    .select(sym_cols)
    .slice(1)                                              # (a) drop warm-up row from shift(1)
    .with_columns(pl.all().exclude("ts_event").fill_null(0.0))  # (c) zero = mean of a demeaned series
    .to_numpy()
)

R=(
    daily_ret
    .select(sym_cols)
    .slice(1)
    .to_numpy()
)
name_vol = R.std(axis=0, ddof=1)                        # (p,)  sigma[i]  in return units
name_vol = np.maximum(name_vol, CAUSAL_VOL_FLOOR)       # floor

lo, hi = np.percentile(name_vol, [2.5, 97.5])
name_vol = np.clip(name_vol, lo, hi)

In [9]:
# Best rank k approximation of the returns covariance matrix
U, s, Vt = np.linalg.svd(X)
k = 10
U_k, S_k, Vt_k = U[:, :k], np.diag(s[:k]), Vt[:k, :]
F = U_k
L = S_k @ Vt_k

# Residuals: what the market factor can't explain
# Adding a floor to epsilon volatility for illiquid names
eps = X - F @ L          # (T, p)
eps_var  = np.var(eps, axis=0)
EPS_FLOOR = 0.1
eps_var  = np.maximum(eps_var, EPS_FLOOR)
D = np.diag(eps_var)         # (2000, 2000) diagonal

# Factor covariance
Sigma_f = np.cov(F, rowvar=False)       # (10, 10)

# Model-implied return covariance: Σ_r = L^T Σ_f L + D
Sigma_r = L.T @ Sigma_f @ L + D         # (2000, 2000)


In [10]:
S = np.diag(name_vol)
L_ret   = L @ S                      # (k, p)  return-unit loadings  B
Dz_diag = eps_var                    # standardized idio var (what you already floor)
Dr_diag = name_vol**2 * Dz_diag      # return-unit idio var  (fallback until GARCH)
Sigma_r_ret = L_ret.T @ Sigma_f @ L_ret + np.diag(Dr_diag)

In [11]:
Sigma_r_ret

array([[0.00161039, 0.00032753, 0.00025634, ..., 0.00076153, 0.00020634,
        0.00024945],
       [0.00032753, 0.00222867, 0.00033068, ..., 0.00027387, 0.00022412,
        0.00112808],
       [0.00025634, 0.00033068, 0.00044425, ..., 0.00021654, 0.00018368,
        0.00029735],
       ...,
       [0.00076153, 0.00027387, 0.00021654, ..., 0.00117197, 0.00026292,
        0.0002864 ],
       [0.00020634, 0.00022412, 0.00018368, ..., 0.00026292, 0.00062814,
        0.00030906],
       [0.00024945, 0.00112808, 0.00029735, ..., 0.0002864 , 0.00030906,
        0.00225138]], shape=(100, 100))

In [12]:
d = np.diag(Sigma_r_ret)
print("diag  min/median/max:", d.min(), np.median(d), d.max())   # want median ≈ 1, max single digits
print(f"cond: {np.linalg.cond(Sigma_r_ret):,.0f}")                          # want O(1e1–1e3)

diag  min/median/max: 0.00011651191694651581 0.0007418115540020037 0.005139492807919411
cond: 390


In [13]:
Sigma_r - (L.T @ Sigma_f @ L + D)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(100, 100))

In [14]:
np.save("name_vol.npy",              name_vol)
np.save("loadings_return.npy",       L_ret)
np.save("idiosyncratic_diag_return.npy", Dr_diag)
np.save("cov_matrix_return.npy",     Sigma_r_ret)

In [15]:
# np.save("loadings.npy", L)
# np.save("factor_returns.npy", F)
# np.save("idiosyncratic_diag_vec.npy", eps_var)
# np.save("factor_cov_matrix.npy", Sigma_f)
# np.save("cov_matrix.npy", Sigma_r)
